Run companion setup SQL in SQLCMD mode with private environment inputs before running query cells. See repository SECRET-SCANNING.md. Saved outputs are cleared.


# SQL Server 2019 Polybase - Data Virtualization
This notebook has a series of instructions to setup external data sources using Polybase in SQL Server 2019

## SQL Server Polybase with Oracle
The following will go through how to setup an external data source, external table, and query to Oracle. First change context to the WideWorldImporters database and remove an existing external tables from a previous run.

### Step 1: Add a master key

Add a master key to encrypt the database scoped credential

In [ ]:
-- Runtime credentials: SQLCMD variables; never save supplied values in source.
IF N'$(MasterKeyPassword)' = N'$' + N'(MasterKeyPassword)' OR LEN(N'$(MasterKeyPassword)') = 0
BEGIN
    RAISERROR('Supply MasterKeyPassword using SQLCMD mode. See SECRET-SCANNING.md. Nothing was changed.', 16, 1);
    SET NOEXEC ON;
END;
GO
CREATE MASTER KEY ENCRYPTION BY PASSWORD = '$(MasterKeyPassword)'
GO
GO
SET NOEXEC OFF;
GO


### Step 2: Create database scoped credentials

In [ ]:
-- Runtime credentials: SQLCMD variables; never save supplied values in source.
IF N'$(ExternalSecret)' = N'$' + N'(ExternalSecret)' OR LEN(N'$(ExternalSecret)') = 0
BEGIN
    RAISERROR('Supply ExternalSecret using SQLCMD mode. See SECRET-SCANNING.md. Nothing was changed.', 16, 1);
    SET NOEXEC ON;
END;
GO
/*  specify credentials to external data source
*  IDENTITY: user name for external source.  
*  SECRET: password for external source.
*/
DROP DATABASE SCOPED CREDENTIAL OracleCredentials
GO
CREATE DATABASE SCOPED CREDENTIAL OracleCredentials   
WITH IDENTITY = 'gl', Secret = '$(ExternalSecret)'
GO
GO
SET NOEXEC OFF;
GO


### 

### Step 3: Create the external data source

In [ ]:
/*  LOCATION: Location string should be of format '<vendor>://<server>[:<port>]'.
*  PUSHDOWN: specify whether computation should be pushed down to the source. ON by default.
*  CREDENTIAL: the database scoped credential, created above.
*/  
CREATE EXTERNAL DATA SOURCE OracleServer
WITH ( 
LOCATION = 'oracle://bworacle:49161',
PUSHDOWN = ON,
CREDENTIAL = OracleCredentials
)
GO


### Step 4: Create a new schema

In [ ]:
DROP SCHEMA oracle
go
CREATE SCHEMA oracle
GO

### Step 5: Create the exteranl table

In [ ]:
/*  LOCATION: oracle table/view in 'database_name.schema_name.object_name' format
*  DATA_SOURCE: the external data source, created above.
*/
CREATE EXTERNAL TABLE oracle.accountsreceivable
(
arid int,
ardate date,
ardesc nvarchar(100) COLLATE Latin1_General_100_CI_AS,
arref int,
aramt decimal(10,2)
)
 WITH (
 LOCATION='[XE].[GL].[ACCOUNTSRECEIVABLE]',
 DATA_SOURCE=OracleServer
)

### Step 6: Create statistics on key columns

In [ ]:
CREATE STATISTICS arrefstats ON oracle.accountsreceivable ([arref]) WITH FULLSCAN
GO

### Step 7: Do a filter query for predicate pushdown

In [ ]:
-- Try a simple filter
SELECT * FROM oracle.accountsreceivable
WHERE arref = 336252
GO

### Step 8: Join with a local table

In [ ]:
-- Join with a local table
--
SELECT ct.*, oa.arid, oa.ardesc
FROM oracle.accountsreceivable oa
JOIN [Sales].[CustomerTransactions] ct
ON oa.arref = ct.CustomerTransactionID
GO